# Phase 1: Provider Layer + write_memory Tool — Production Code Tests

Testing the production modules:
- `oai_context.py` — Context object for OpenAI Agents SDK
- `oai_providers.py` — LitellmModel builders from lerim config
- `oai_tools.py` — write_memory, extract_pipeline, summarize_pipeline

In [8]:
import os, sys, json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("/Users/kargarisaac/codes/personal/lerim/lerim-cli/.env"))
sys.path.insert(0, str(Path("/Users/kargarisaac/codes/personal/lerim/lerim-cli/.claude/worktrees/agent-af7d097c/src")))

from lerim.runtime.oai_context import OAIRuntimeContext, build_oai_context
from lerim.runtime.oai_providers import build_oai_model, build_oai_model_from_role, build_oai_fallback_models
from lerim.runtime.oai_tools import write_memory, extract_pipeline, summarize_pipeline
from agents import Agent, Runner, set_tracing_disabled
set_tracing_disabled(disabled=True)
print("All imports successful")

All imports successful


## Test 1 — Context: `build_oai_context`

Create a context with temp dirs, verify all fields resolve correctly.

In [9]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
	root = Path(tmp)
	memory_root = root / "memory"
	memory_root.mkdir()
	workspace = root / "workspace"
	workspace.mkdir()
	extra = root / "extra_docs"
	extra.mkdir()

	ctx = build_oai_context(
		repo_root=root,
		memory_root=memory_root,
		workspace_root=workspace,
		run_folder=workspace / "run-001",
		extra_read_roots=(str(extra),),
		run_id="test-20260325-120000-abc1",
	)

	# Verify all fields
	assert isinstance(ctx, OAIRuntimeContext), f"Expected OAIRuntimeContext, got {type(ctx)}"
	assert ctx.repo_root == root.resolve(), f"repo_root mismatch: {ctx.repo_root}"
	assert ctx.memory_root == memory_root.resolve(), f"memory_root mismatch: {ctx.memory_root}"
	assert ctx.workspace_root == workspace.resolve(), f"workspace_root mismatch: {ctx.workspace_root}"
	assert ctx.run_folder == (workspace / "run-001").resolve(), f"run_folder mismatch: {ctx.run_folder}"
	assert len(ctx.extra_read_roots) == 1, f"extra_read_roots count: {len(ctx.extra_read_roots)}"
	assert ctx.extra_read_roots[0] == extra.resolve(), f"extra_read_roots[0] mismatch"
	assert ctx.run_id == "test-20260325-120000-abc1", f"run_id mismatch: {ctx.run_id}"
	assert ctx.config is not None, "config should be populated"
	assert ctx.trace_path is None, "trace_path should be None when not provided"
	assert ctx.artifact_paths is None, "artifact_paths should be None when not provided"

	print(f"repo_root:       {ctx.repo_root}")
	print(f"memory_root:     {ctx.memory_root}")
	print(f"workspace_root:  {ctx.workspace_root}")
	print(f"run_folder:      {ctx.run_folder}")
	print(f"extra_read_roots: {ctx.extra_read_roots}")
	print(f"run_id:          {ctx.run_id}")
	print(f"config.provider: {ctx.config.provider}")
	print("\nTest 1 PASSED: build_oai_context works correctly")

repo_root:       /private/var/folders/yw/4fydhsnj05x2lndgxkn2jb_80000gn/T/tmp5knrdymp
memory_root:     /private/var/folders/yw/4fydhsnj05x2lndgxkn2jb_80000gn/T/tmp5knrdymp/memory
workspace_root:  /private/var/folders/yw/4fydhsnj05x2lndgxkn2jb_80000gn/T/tmp5knrdymp/workspace
run_folder:      /private/var/folders/yw/4fydhsnj05x2lndgxkn2jb_80000gn/T/tmp5knrdymp/workspace/run-001
extra_read_roots: (PosixPath('/private/var/folders/yw/4fydhsnj05x2lndgxkn2jb_80000gn/T/tmp5knrdymp/extra_docs'),)
run_id:          test-20260325-120000-abc1
config.provider: minimax

Test 1 PASSED: build_oai_context works correctly


## Test 2 — Providers: `build_oai_model`

Build the lead model from config. Print provider, model name, type.
Then build with MiniMax directly from a `LLMRoleConfig`.

In [10]:
from lerim.config.settings import get_config, LLMRoleConfig
from agents.extensions.models.litellm_model import LitellmModel

cfg = get_config()

# Build lead model from config
model = build_oai_model("lead", config=cfg)
assert isinstance(model, LitellmModel), f"Expected LitellmModel, got {type(model)}"
print(f"Lead provider:  {cfg.lead_role.provider}")
print(f"Lead model:     {cfg.lead_role.model}")
print(f"LitellmModel:   {type(model).__name__}")
print(f"Model string:   {model.model}")

# Build explorer model from config
explorer_model = build_oai_model("explorer", config=cfg)
assert isinstance(explorer_model, LitellmModel)
print(f"\nExplorer provider: {cfg.explorer_role.provider}")
print(f"Explorer model:    {cfg.explorer_role.model}")

# Build directly from a MiniMax LLMRoleConfig
minimax_role = LLMRoleConfig(
	provider="minimax",
	model="MiniMax-M2.5",
	api_base="",
	fallback_models=(),
	timeout_seconds=120,
	max_iterations=10,
	openrouter_provider_order=(),
)
minimax_model = build_oai_model_from_role(minimax_role, config=cfg)
assert isinstance(minimax_model, LitellmModel)
assert minimax_model.model == "minimax/MiniMax-M2.5"
print(f"\nMiniMax model string: {minimax_model.model}")

# Build fallback models from lead config
fallbacks = build_oai_fallback_models(cfg.lead_role, config=cfg)
print(f"\nFallback count: {len(fallbacks)}")
for i, fb in enumerate(fallbacks):
	print(f"  fallback[{i}]: {fb.model}")

print("\nTest 2 PASSED: Provider builders work correctly")

Lead provider:  minimax
Lead model:     MiniMax-M2.5
LitellmModel:   LitellmModel
Model string:   minimax/MiniMax-M2.5

Explorer provider: minimax
Explorer model:    MiniMax-M2.5

MiniMax model string: minimax/MiniMax-M2.5

Fallback count: 1
  fallback[0]: openai/glm-4.7

Test 2 PASSED: Provider builders work correctly


## Test 3 — Provider connects: live LLM call

Create a simple agent with the built model, ask "what is 2+2", verify response.

In [11]:
from lerim.config.settings import get_config

cfg = get_config()
model = build_oai_model("lead", config=cfg)

agent = Agent(
	name="MathTest",
	instructions="You are a math assistant. Answer concisely with just the number.",
	model=model,
)

result = await Runner.run(agent, "What is 2+2?")
print(f"Agent response: {result.final_output}")
assert "4" in result.final_output, f"Expected '4' in response, got: {result.final_output}"
print("\nTest 3 PASSED: Provider connects and returns valid response")

Agent response: 

4

Test 3 PASSED: Provider connects and returns valid response


## Test 4 — `write_memory` tool via agent

Create an agent with write_memory tool and the built model.
Ask it to save a decision. Verify the file was written with correct frontmatter.

In [12]:
import tempfile
import frontmatter
from pathlib import Path
from lerim.config.settings import get_config

cfg = get_config()
model = build_oai_model("lead", config=cfg)

with tempfile.TemporaryDirectory() as tmp:
	root = Path(tmp)
	memory_root = root / "memory"
	(memory_root / "decisions").mkdir(parents=True)
	(memory_root / "learnings").mkdir(parents=True)

	ctx = build_oai_context(
		repo_root=root,
		memory_root=memory_root,
		run_id="test-20260325-140000-def2",
		config=cfg,
	)

	agent = Agent(
		name="MemoryWriter",
		instructions=(
			"You are a test agent. Use the write_memory tool to save a decision with "
			"primitive='decision', title='Use SQLite for session indexing', "
			"body='SQLite provides ACID guarantees with zero deployment overhead. "
			"Chose it over PostgreSQL for the local-first architecture.', "
			"confidence=0.9, tags='database,architecture'. "
			"Then respond with ONLY the JSON result from the tool, nothing else."
		),
		model=model,
		tools=[write_memory],
	)

	result = await Runner.run(agent, "Save the memory as instructed.", context=ctx)
	print(f"Agent response: {result.final_output[:300]}")

	# Verify file was written
	written = list(memory_root.rglob("*.md"))
	assert len(written) > 0, "No memory file was written"

	mem_file = written[0]
	print(f"\nFile written: {mem_file.name}")
	print(f"File path:    {mem_file}")

	# Parse and verify frontmatter
	post = frontmatter.load(str(mem_file))
	print(f"\nFrontmatter keys: {list(post.metadata.keys())}")
	print(f"Title:      {post.metadata.get('title')}")
	print(f"Confidence: {post.metadata.get('confidence')}")
	print(f"Tags:       {post.metadata.get('tags')}")
	print(f"Source:     {post.metadata.get('source')}")
	print(f"Body:       {post.content[:100]}...")

	assert post.metadata.get("title") == "Use SQLite for session indexing"
	assert post.metadata.get("confidence") == 0.9
	assert "database" in post.metadata.get("tags", [])
	assert "architecture" in post.metadata.get("tags", [])
	assert post.metadata.get("source") == "test-20260325-140000-def2"
	assert "SQLite" in post.content or "sqlite" in post.content.lower()

	print("\nTest 4 PASSED: write_memory tool creates correct file with frontmatter")

Agent response: 

{"file_path": "/private/var/folders/yw/4fydhsnj05x2lndgxkn2jb_80000gn/T/tmp_omny4uh/memory/decisions/20260325-use-sqlite-for-session-indexing.md", "bytes": 376, "primitive": "decision"}

File written: 20260325-use-sqlite-for-session-indexing.md
File path:    /var/folders/yw/4fydhsnj05x2lndgxkn2jb_80000gn/T/tmp_omny4uh/memory/decisions/20260325-use-sqlite-for-session-indexing.md

Frontmatter keys: ['confidence', 'created', 'id', 'source', 'tags', 'title', 'updated']
Title:      Use SQLite for session indexing
Confidence: 0.9
Tags:       ['database', 'architecture']
Source:     test-20260325-140000-def2
Body:       SQLite provides ACID guarantees with zero deployment overhead. Chose it over PostgreSQL for the loca...

Test 4 PASSED: write_memory tool creates correct file with frontmatter


## Test 5 — `write_memory` validation: error handling

Create an agent and ask it to save a memory with invalid parameters.
The agent should receive the error and self-correct.
Test: invalid primitive, missing kind on learning.

In [13]:
import tempfile
from pathlib import Path
from lerim.config.settings import get_config

cfg = get_config()
model = build_oai_model("lead", config=cfg)

with tempfile.TemporaryDirectory() as tmp:
	root = Path(tmp)
	memory_root = root / "memory"
	(memory_root / "decisions").mkdir(parents=True)
	(memory_root / "learnings").mkdir(parents=True)

	ctx = build_oai_context(
		repo_root=root,
		memory_root=memory_root,
		run_id="test-20260325-150000-ghi3",
		config=cfg,
	)

	# Test: agent starts with invalid primitive, should self-correct
	agent = Agent(
		name="ValidationTest",
		instructions=(
			"You are a test agent. First, try to call write_memory with "
			"primitive='note' (which is invalid). When you get an error, "
			"self-correct and use primitive='learning' with "
			"title='Retry logic matters', body='Always implement retry with backoff.', "
			"confidence=0.7, tags='reliability', kind='insight'. "
			"Respond with ONLY the final JSON result from the successful tool call."
		),
		model=model,
		tools=[write_memory],
	)

	result = await Runner.run(agent, "Save the memory as instructed.", context=ctx)
	print(f"Agent response: {result.final_output[:300]}")

	# The agent should have self-corrected and written a valid file
	written = list(memory_root.rglob("*.md"))
	assert len(written) > 0, "Agent failed to self-correct; no file written"
	print(f"\nFile written after self-correction: {written[0].name}")

	# Verify it is in the learnings folder
	assert "learnings" in str(written[0]), f"Expected learnings folder, got: {written[0]}"

	# Parse and verify kind is present
	import frontmatter
	post = frontmatter.load(str(written[0]))
	assert post.metadata.get("kind") == "insight", f"Expected kind='insight', got: {post.metadata.get('kind')}"
	print(f"Kind: {post.metadata.get('kind')}")

	print("\nTest 5 PASSED: Agent self-corrects on validation errors")

Agent response: 

{"file_path": "/private/var/folders/yw/4fydhsnj05x2lndgxkn2jb_80000gn/T/tmpky1p668k/memory/learnings/20260325-retry-logic-matters.md", "bytes": 269, "primitive": "learning"}

File written after self-correction: 20260325-retry-logic-matters.md
Kind: insight

Test 5 PASSED: Agent self-corrects on validation errors


## Test 6 — Combined: Codex + write_memory

Create test memories in a temp dir. Create an agent with both
`codex_tool` (read-only) and `write_memory`. Ask it to read existing
memories and create a new one that does not duplicate.

In [14]:
import tempfile
from pathlib import Path
from lerim.config.settings import get_config
from lerim.runtime.oai_providers import build_codex_options
from lerim.runtime.responses_proxy import ResponsesProxy
from agents.extensions.experimental.codex import codex_tool, ThreadOptions, TurnOptions, CodexOptions

cfg = get_config()
model = build_oai_model("lead", config=cfg)

with tempfile.TemporaryDirectory() as tmp:
	root = Path(tmp)
	memory_root = root / "memory"
	(memory_root / "decisions").mkdir(parents=True)
	(memory_root / "learnings").mkdir(parents=True)

	# Seed existing memories
	existing_decision = memory_root / "decisions" / "20260325-use-sqlite.md"
	existing_decision.write_text(
		"---\n"
		"id: use-sqlite\n"
		"title: Use SQLite for indexing\n"
		"created: 2026-03-25T10:00:00+00:00\n"
		"updated: 2026-03-25T10:00:00+00:00\n"
		"source: seed\n"
		"confidence: 0.9\n"
		"tags: [database, architecture]\n"
		"---\n"
		"SQLite for local-first session indexing.\n",
		encoding="utf-8",
	)

	existing_learning = memory_root / "learnings" / "20260325-queue-heartbeat.md"
	existing_learning.write_text(
		"---\n"
		"id: queue-heartbeat\n"
		"title: Queue heartbeat pattern\n"
		"created: 2026-03-25T10:00:00+00:00\n"
		"updated: 2026-03-25T10:00:00+00:00\n"
		"source: seed\n"
		"kind: insight\n"
		"confidence: 0.8\n"
		"tags: [queue, reliability]\n"
		"---\n"
		"Keep heartbeat updates deterministic.\n",
		encoding="utf-8",
	)

	ctx = build_oai_context(
		repo_root=root,
		memory_root=memory_root,
		run_id="test-20260325-160000-jkl4",
		config=cfg,
	)

	# Build codex options from lerim config — handles proxy automatically
	codex_opts, thread_opts, needs_proxy = build_codex_options(config=cfg)
	print(f"Provider: {cfg.lead_role.provider}, needs_proxy: {needs_proxy}")

	proxy = None
	if needs_proxy:
		proxy = ResponsesProxy(
			backend_url=codex_opts["backend_url"],
			api_key=codex_opts["backend_api_key"],
		)
		proxy.start()
		codex_opts["base_url"] = proxy.url
		codex_opts["api_key"] = "proxy-managed"
		print(f"Proxy started at {proxy.url} → {codex_opts['backend_url']}")

	try:
		# Remove proxy-internal keys before passing to CodexOptions
		clean_codex_opts = {k: v for k, v in codex_opts.items() if k in ("base_url", "api_key") and v is not None}

		codex = codex_tool(
			codex_options=CodexOptions(**clean_codex_opts),
			sandbox_mode="read-only",
			working_directory=str(root),
			skip_git_repo_check=True,
			default_thread_options=ThreadOptions(**thread_opts),
		)

		agent = Agent(
			name="CombinedTest",
			instructions=(
				f"You are a memory management agent. The memory root is {memory_root}.\n"
				"First, use the codex tool to list files in the memory/decisions and "
				"memory/learnings folders and read their contents.\n"
				"Then, create ONE new decision memory that does NOT duplicate any existing ones. "
				"Use write_memory with primitive='decision', a unique title about "
				"'Use TOML for configuration', body describing why TOML was chosen over YAML, "
				"confidence=0.85, tags='config,tooling'.\n"
				"Respond with ONLY the final JSON from write_memory."
			),
			model=model,
			tools=[codex, write_memory],
		)

		result = await Runner.run(agent, "Read existing memories and create a new non-duplicate.", context=ctx)
		print(f"\nAgent response: {result.final_output[:300]}")

		# Verify: should now have 3 files total (2 seeded + 1 new)
		all_files = list(memory_root.rglob("*.md"))
		print(f"\nTotal memory files: {len(all_files)}")
		for f in sorted(all_files):
			print(f"  {f.relative_to(memory_root)}")

		decision_files = list((memory_root / "decisions").glob("*.md"))
		assert len(decision_files) >= 2, f"Expected at least 2 decisions, got {len(decision_files)}"

		new_files = [f for f in decision_files if f.name != "20260325-use-sqlite.md"]
		assert len(new_files) > 0, "No new decision file was created"
		print(f"\nNew decision file: {new_files[0].name}")

		import frontmatter
		post = frontmatter.load(str(new_files[0]))
		print(f"Title: {post.metadata.get('title')}")
		print(f"Tags:  {post.metadata.get('tags')}")
		print(f"Codex model: {thread_opts['model']}")

		print("\nTest 6 PASSED: Codex + write_memory work together without duplication")
	finally:
		if proxy:
			proxy.stop()
			print("Proxy stopped")

ImportError: cannot import name 'build_codex_options' from 'lerim.runtime.oai_providers' (/Users/kargarisaac/codes/personal/lerim/lerim-cli/.claude/worktrees/agent-af7d097c/src/lerim/runtime/oai_providers.py)

20:23:15 | INFO     | Exception in execute request:
---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
Cell In[14], line 4
      2 from pathlib import Path
      3 from lerim.config.settings import get_config
----> 4 from lerim.runtime.oai_providers import build_codex_options
      5 from lerim.runtime.responses_proxy import ResponsesProxy
      6 from agents.extensions.experimental.codex import codex_tool, ThreadOptions, TurnOptions, CodexOptions

ImportError: cannot import name 'build_codex_options' from 'lerim.runtime.oai_providers' (/Users/kargarisaac/codes/personal/lerim/lerim-cli/.claude/worktrees/agent-af7d097c/src/lerim/runtime/oai_providers.py)



## Summary

In [ ]:
print("=" * 60)
print("Phase 1 Production Code Tests")
print("=" * 60)
print()
tests = [
	("Test 1", "build_oai_context", "Context fields resolve correctly"),
	("Test 2", "build_oai_model", "Provider builders create LitellmModel instances"),
	("Test 3", "Provider connects", "Live LLM call returns valid response"),
	("Test 4", "write_memory", "Tool creates file with correct frontmatter"),
	("Test 5", "write_memory validation", "Agent self-corrects on invalid params"),
	("Test 6", "Codex + write_memory", "Combined tools avoid duplication"),
]
for num, name, desc in tests:
	print(f"  {num}: {name:.<30s} {desc}")
print()
print("All tests defined. Run cells sequentially after Setup (Cell 2).")
print("=" * 60)